In [ ]:
import torch
import numpy as np
import cv2
from pathlib import Path

class DropletDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir, sigma=3.0, downscale=1):
        self.img_paths = sorted(Path(img_dir).glob("*"))
        self.label_dir = Path(label_dir)
        self.sigma = float(sigma)
        self.downscale = int(downscale)

    def __len__(self):
        return len(self.img_paths)

    @staticmethod
    def _read_gray_u8(path: Path):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Failed to read image: {path}")
        if img.dtype != np.uint8:
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return img

    def _read_coords(self, label_path: Path):
        coords = []
        if label_path.exists():
            with open(label_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 2:
                        continue
                    x, y = float(parts[0]), float(parts[1])
                    coords.append((x, y))
        return coords

    def _add_gaussian_local(self, heatmap, cx, cy):
        """
        Draw gaussian into heatmap locally (crop window ~ 3*sigma).
        heatmap: float32 HxW
        cx, cy: float coords in heatmap pixel space
        """
        H, W = heatmap.shape
        sigma = self.sigma
        rad = int(np.ceil(3.0 * sigma))

        x0 = int(np.floor(cx)) - rad
        x1 = int(np.floor(cx)) + rad + 1
        y0 = int(np.floor(cy)) - rad
        y1 = int(np.floor(cy)) + rad + 1

        # clip to image
        xs0 = max(0, x0); xs1 = min(W, x1)
        ys0 = max(0, y0); ys1 = min(H, y1)
        if xs0 >= xs1 or ys0 >= ys1:
            return

        # local grid
        yy, xx = np.mgrid[ys0:ys1, xs0:xs1].astype(np.float32)
        g = np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2.0 * sigma * sigma)).astype(np.float32)

        # take max (peak-preserving)
        heatmap[ys0:ys1, xs0:xs1] = np.maximum(heatmap[ys0:ys1, xs0:xs1], g)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        img_u8 = self._read_gray_u8(img_path)

        # optional downscale to reduce RAM/compute
        if self.downscale > 1:
            H, W = img_u8.shape
            img_u8 = cv2.resize(img_u8, (W // self.downscale, H // self.downscale), interpolation=cv2.INTER_AREA)

        img = img_u8.astype(np.float32) / 255.0
        H, W = img.shape

        heatmap = np.zeros((H, W), dtype=np.float32)

        coords = self._read_coords(label_path)
        if self.downscale > 1:
            coords = [(x / self.downscale, y / self.downscale) for (x, y) in coords]

        for x, y in coords:
            self._add_gaussian_local(heatmap, x, y)

        # tensors: (1,H,W)
        img_t = torch.from_numpy(img).unsqueeze(0)
        hm_t = torch.from_numpy(heatmap).unsqueeze(0)
        return img_t, hm_t

In [ ]:
from torch.utils.data import DataLoader

dataset = DropletDataset(r"data\NETWORK\images",
                         r"data\NETWORK\labels",
                         sigma=25.0, downscale=2)

loader = DataLoader(
    dataset,
    batch_size=4,          # начни с 1-4
    shuffle=True,
    num_workers=0,         # ВАЖНО на Windows для стабильности
    pin_memory=True,       # ускоряет CPU->GPU
    persistent_workers=False
)

In [ ]:
import torch.nn as nn

class DropletNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 1, 1),
        )

    def forward(self, x):
        return self.net(x)

device = 'cuda'
model = DropletNet().to(device)

# pos_weight можно подобрать: чем реже "положительные" пиксели, тем больше
pos_weight = torch.tensor([30.0], device=device)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


# 2) Model / Loss / Optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DropletNet().to(device)

# pos_weight: чем больше "пустого" фона, тем больше вес
pos_weight = torch.tensor([30.0], device=device)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 3) Optional AMP (экономит VRAM на CUDA)
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# 4) Train loop
epochs = 20
model.train()

for epoch in range(1, epochs + 1):
    total = 0.0
    for imgs, targets in tqdm(loader, desc=f"Epoch {epoch}/{epochs}"):
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)              # (B,1,H,W) logits
            loss = loss_fn(logits, targets)   # targets in [0..1]

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += float(loss.item())

    print(f"Epoch {epoch}: loss={total/len(loader):.6f}")

# сохранить модель
torch.save(model.state_dict(), "droplet_net.pt")
print("Saved droplet_net.pt")

In [ ]:
import numpy as np
import torch
import cupy as cp  # не обязательно; тут всё CPU
from scipy.ndimage import maximum_filter

def greedy_nms_grid_xy(xy: np.ndarray, scores: np.ndarray, min_distance: int) -> np.ndarray:
    """
    xy: (M,2) int (x,y)
    scores: (M,)
    returns: (N,2) int (x,y)
    """
    if xy.shape[0] == 0:
        return xy

    order = np.argsort(scores)[::-1]
    xy = xy[order]

    cell = max(1, int(min_distance))
    min_d2 = int(min_distance) ** 2

    grid = {}  # (cy,cx)-> list[(x,y)]
    out = []

    for x, y in xy:
        cx, cy = x // cell, y // cell
        ok = True
        for gx in (cx - 1, cx, cx + 1):
            for gy in (cy - 1, cy, cy + 1):
                pts = grid.get((gx, gy))
                if not pts:
                    continue
                for px, py in pts:
                    dx = int(x - px)
                    dy = int(y - py)
                    if dx*dx + dy*dy < min_d2:
                        ok = False
                        break
                if not ok:
                    break
            if not ok:
                break
        if ok:
            out.append((int(x), int(y)))
            grid.setdefault((cx, cy), []).append((int(x), int(y)))

    return np.asarray(out, dtype=np.int32)

def heatmap_to_points(
    heatmap: np.ndarray,
    *,
    thr: float = 0.3,
    min_distance: int = 8,
    max_points: int = 5000,
):
    """
    heatmap: (H,W) float32 in [0..1]
    Returns:
      xy: (N,2) int32 (x,y)
      scores: (N,) float32
    """
    H, W = heatmap.shape

    # локальные максимумы
    win = 2*int(min_distance) + 1
    maxf = maximum_filter(heatmap, size=win, mode="constant", cval=-np.inf)
    cand = (heatmap >= (maxf - 1e-8)) & (heatmap >= float(thr))

    ys, xs = np.nonzero(cand)
    if ys.size == 0:
        return np.zeros((0,2), np.int32), np.zeros((0,), np.float32)

    scores = heatmap[ys, xs].astype(np.float32, copy=False)

    # ограничим кандидатов
    if ys.size > max_points:
        k = int(max_points)
        idx = np.argpartition(scores, -k)[-k:]
        xs = xs[idx]; ys = ys[idx]; scores = scores[idx]

    xy = np.stack([xs, ys], axis=1).astype(np.int32)

    # точный greedy NMS
    xy_out = greedy_nms_grid_xy(xy, scores, min_distance=min_distance)

    # пересчёт scores для оставшихся точек
    sc_out = heatmap[xy_out[:,1], xy_out[:,0]].astype(np.float32, copy=False)
    return xy_out, sc_out

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

# загрузить веса
model = DropletNet().to(device)
model.load_state_dict(torch.load("droplet_net.pt", map_location=device))
model.eval()

# параметры соответствуют dataset.downscale
DOWNSCALE = 2

img_path = 
img_u8 = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
H0, W0 = img_u8.shape

# тот же даунскейл, что на обучении
if DOWNSCALE > 1:
    img_u8_ds = cv2.resize(img_u8, (W0//DOWNSCALE, H0//DOWNSCALE), interpolation=cv2.INTER_AREA)
else:
    img_u8_ds = img_u8

img = (img_u8_ds.astype(np.float32)/255.0)[None, None, :, :]  # (1,1,H,W)
img_t = torch.from_numpy(img).to(device)

with torch.no_grad():
    logits = model(img_t)
    heat = torch.sigmoid(logits)[0,0].detach().cpu().numpy().astype(np.float32)


In [ ]:

# в координатах downscale-сетки
xy_ds, sc = heatmap_to_points(heat, thr=0.55, min_distance=30)

# вернуть в оригинальные координаты
xy_full = xy_ds.astype(np.int32) * DOWNSCALE

# визуализация
disp = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
for (x, y) in xy_full:
    cv2.drawMarker(disp, (int(x), int(y)), (0,0,255), cv2.MARKER_CROSS, 14, 2, cv2.LINE_AA)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(img_u8, cmap="gray")
plt.title("Input")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(disp, cv2.COLOR_BGR2RGB))
plt.title(f"Predicted points: {len(xy_full)}")
plt.axis("off")
plt.tight_layout()
plt.savefig(r"data\pumpQ_figures\NETZWERK.png")
plt.show()

In [ ]:
state = torch.load("droplet_net.pt")

for k, v in state.items():
    print(k, v.shape)

In [ ]:
weights = state["net.0.weight"]   # имя слоя из вывода выше
weights = weights.cpu()
weights = weights.detach().numpy()

n_filters = weights.shape[0]

plt.figure(figsize=(10,6))

for i in range(n_filters):
    plt.subplot(4,4,i+1)
    plt.imshow(weights[i,0], cmap="gray")
    plt.axis("off")
    plt.title(f"F{i}")

plt.suptitle("Convolution layer filters")
plt.savefig(r"data\pumpQ_figures\Convolution.png")
plt.show()

In [ ]:
import torch
import cv2
import numpy as np
import time
from pathlib import Path

# -----------------------------
# ПАРАМЕТРЫ
# -----------------------------

image_folder = r"data\NETWORK\images"
model_path = r"data\droplet_net.pt"


device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# ЗАГРУЗКА МОДЕЛИ
# -----------------------------

model = DropletNet().to(device)
model.load_state_dict(torch.load("droplet_net.pt", map_location=device))
model.eval()

# -----------------------------
# СПИСОК ФАЙЛОВ
# -----------------------------

image_paths = sorted(list(Path(image_folder).glob("*.*")))

print("Images found:", len(image_paths))

# -----------------------------
# ИЗМЕРЕНИЕ СКОРОСТИ
# -----------------------------

times = []

with torch.no_grad():

    for path in image_paths:

        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # нормализация
        img = img.astype(np.float32) / 255.0

        # tensor
        tensor = torch.from_numpy(img)[None, None, :, :]
        tensor = tensor.to(device)

        start = time.perf_counter()

        pred = model(tensor)

        if device == "cuda":
            torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

# -----------------------------
# СТАТИСТИКА
# -----------------------------

times = np.array(times)

print("\nDetection timing statistics")
print("---------------------------")
print("Frames processed:", len(times))
print("Mean time per frame:", times.mean(), "s")
print("Median time:", np.median(times), "s")
print("Std:", times.std(), "s")
print("FPS:", 1 / times.mean())

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import median_filter

def extract_histogram_from_png(
    png_path: str,
    x_range=(0.0, 60.0),      # как на твоей оси X
    y_tick_step=5,            # у тебя видны линии 0,5,10,15,20,25
    blue_hsv_lo=(90, 50, 50), # диапазон синего для matplotlib
    blue_hsv_hi=(140, 255, 255),
    smooth_win=5,
):
    img = cv2.imread(png_path)
    if img is None:
        raise FileNotFoundError(png_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)

    # --- 1) найти рамку осей/поля графика по длинным линиям ---
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=80, minLineLength=200, maxLineGap=5)
    if lines is None:
        raise RuntimeError("Hough lines not found")

    xs0, xs1, ys0, ys1 = [], [], [], []
    for (x0, y0, x1, y1) in lines[:, 0, :]:
        xs0.append(x0); xs1.append(x1)
        ys0.append(y0); ys1.append(y1)

    # грубо: берем экстремумы по вертикальным и горизонтальным линиям
    # (это работает на типичных matplotlib PNG)
    x_min = min(min(xs0), min(xs1))
    x_max = max(max(xs0), max(xs1))
    y_min = min(min(ys0), min(ys1))
    y_max = max(max(ys0), max(ys1))

    # небольшой safety-clip
    H, W = gray.shape
    x0 = max(0, x_min)
    x1 = min(W-1, x_max)
    y0 = max(0, y_min)
    y1 = min(H-1, y_max)

    plot_bgr = img[y0:y1, x0:x1]
    plot_gray = gray[y0:y1, x0:x1]

    # --- 2) маска синего (столбики) ---
    plot_hsv = cv2.cvtColor(plot_bgr, cv2.COLOR_BGR2HSV)
    mask_blue = cv2.inRange(plot_hsv, blue_hsv_lo, blue_hsv_hi) > 0

    # --- 3) калибровка Y: найти горизонтальные gridlines (после удаления синего) ---
    plot_gray_clean = plot_gray.copy()
    plot_gray_clean[mask_blue] = 255

    row_mean = plot_gray_clean.mean(axis=1)
    row_std  = plot_gray_clean.std(axis=1)

    # gridline — темнее фона и “однородная” по X
    cand = np.where((row_mean < 250) & (row_std < 15))[0]

    # кластеризуем по непрерывным диапазонам
    clusters = []
    if cand.size:
        s = cand[0]; p = cand[0]
        for r in cand[1:]:
            if r == p + 1:
                p = r
            else:
                clusters.append((s, p))
                s = p = r
        clusters.append((s, p))

    # уберём верхнюю рамку, оставим внутренние линии
    grid_rows = []
    for a, b in clusters:
        # верхняя рамка обычно очень близко к y=0
        if a <= 3:
            continue
        grid_rows.append(int((a + b) // 2))
    grid_rows = sorted(grid_rows)

    # baseline (0) — нижняя чёрная ось, примерно низ ROI
    baseline_row = plot_gray.shape[0] - 1

    # в твоём примере видно 5 линий: 25,20,15,10,5 (плюс baseline=0)
    # если линий меньше/больше — можно поправить вручную
    if len(grid_rows) < 3:
        raise RuntimeError(f"Too few grid lines detected: {grid_rows}")

    # возьмём последние 5 линий как 25..5 (если больше — обрезаем)
    grid_rows = grid_rows[-5:]
    tick_vals = np.array([25, 20, 15, 10, 5], dtype=float)
    tick_y    = np.array([y0 + r for r in grid_rows], dtype=float)

    # линейная регрессия: y_pix = a + b * val
    A = np.vstack([np.ones_like(tick_vals), tick_vals]).T
    a, b = np.linalg.lstsq(A, tick_y, rcond=None)[0]

    def ypix_to_count(y_pix: float) -> float:
        # val = (y - a) / b
        return float((y_pix - a) / b)

    # --- 4) профиль высоты по X ---
    Hh, Ww = mask_blue.shape
    top_row = np.full((Ww,), baseline_row, dtype=int)
    has = np.zeros(Ww, dtype=bool)
    for x in range(Ww):
        ys = np.where(mask_blue[:, x])[0]
        if ys.size:
            has[x] = True
            top_row[x] = int(ys.min())

    y_abs = y0 + top_row
    counts_est = np.array([ypix_to_count(y) for y in y_abs], dtype=float)
    counts_est[~has] = 0.0

    # сгладим и округлим к целым counts
    counts_s = median_filter(counts_est, size=int(smooth_win), mode="nearest")
    counts_i = np.rint(counts_s).astype(int)
    counts_i[counts_i < 0] = 0

    # --- 5) собрать "видимые" прямоугольники: где count>0 и меняется ---
    x_min_data, x_max_data = map(float, x_range)
    def xplot_to_data(x_plot):
        return x_min_data + (x_plot / (Ww - 1)) * (x_max_data - x_min_data)

    bars = []
    x = 0
    while x < Ww:
        if counts_i[x] == 0:
            x += 1
            continue
        c = int(counts_i[x])
        start = x
        while x + 1 < Ww and counts_i[x + 1] == c:
            x += 1
        end = x

        left  = xplot_to_data(start - 0.5)
        right = xplot_to_data(end + 0.5)
        center = xplot_to_data((start + end) / 2)

        bars.append((center, left, right, c))
        x += 1

    center_um = np.array([b[0] for b in bars], dtype=float)
    left_um   = np.array([b[1] for b in bars], dtype=float)
    right_um  = np.array([b[2] for b in bars], dtype=float)
    counts    = np.array([b[3] for b in bars], dtype=int)

    return center_um, counts, left_um, right_um


# --- пример использования ---
png_path = "sad_hist.png"
center_um, counts, left_um, right_um = extract_histogram_from_png(png_path)

print("centers_um =", center_um.tolist())
print("counts     =", counts.tolist())

In [ ]:
import matplotlib.pyplot as plt
plt.plot(centers_um, counts)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.ndimage import median_filter
%matplotlib widget

centers_um = np.array([
  11.588447653429602, 14.24187725631769, 16.570397111913355, 17.65342960288809,
  19.223826714801444, 20.902527075812277, 21.60649819494585, 22.256317689530686,
  22.906137184115522, 23.610108303249095, 24.25992779783393, 24.855595667870038,
  25.55956678700361, 26.263537906137184, 26.913357400722022, 27.617328519855597,
  28.32129963898917, 28.971119133574007, 29.620938628158844, 30.32490974729242,
  30.974729241877257, 31.949458483754512, 33.5740072202166, 36.60649819494585,
  38.23104693140794, 43.64620938628159
], dtype=float)

counts = np.array([
  2, 2, 2, 4, 2, 6, 2, 6, 12, 20, 28, 16, 20, 16, 22, 16, 12, 6, 14, 16, 10, 2, 2, 2, 2, 2
], dtype=float)

plt.rcParams.update({"figure.dpi": 100})

def gauss_counts(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

x = centers_um
y = counts

A0 = float(y.max())
mu0 = float(np.average(x, weights=y))
sigma0 = float(np.sqrt(np.average((x - mu0) ** 2, weights=y)))

popt, _ = curve_fit(
    gauss_counts, x, y,
    p0=[A0, mu0, sigma0],
    bounds=([0, -np.inf, 1e-6], [np.inf, np.inf, np.inf])
)
A_fit, mu_fit, sigma_fit = map(float, popt)

y_hat = gauss_counts(x, *popt)
resid = y - y_hat

ss_res = float(np.sum(resid**2))
ss_tot = float(np.sum((y - y.mean())**2))
r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

# 1) Гистограмма + фит
dx = np.diff(np.sort(centers_um))
bar_w = float(np.median(dx)) if dx.size else 1.0

plt.figure(figsize=(7.2, 4.2))
plt.bar(centers_um, counts, width=0.85*bar_w, alpha=0.8, edgecolor="none", label="Counts of droplet sizes")
xx = np.linspace(centers_um.min()-2*bar_w, centers_um.max()+2*bar_w, 600)
plt.plot(xx, gauss_counts(xx, *popt), linewidth=2.0, label="Gaussian fit")
plt.title("Droplet size histogram + Gaussian fit; mu=26.1 µm, sigma=4.1 µm; R^2=0.83")
plt.xlabel("Linear size (µm)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 2) Остатки
plt.figure(figsize=(7.2, 3.6))
plt.axhline(0.0, linewidth=1.0)
plt.scatter(centers_um, resid, s=35)
plt.title("Fit residuals (count - fitted)")
plt.xlabel("Linear size (µm)")
plt.ylabel("Residual (counts)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3) Параметры по “восстановленным сэмплам”
samples = np.repeat(centers_um, counts.astype(int))
mu_mle = float(np.mean(samples))
sigma_mle = float(np.std(samples, ddof=0))

print("Gaussian fit to binned counts:")
print(f"  A={A_fit:.3f}, mu={mu_fit:.3f} µm, sigma={sigma_fit:.3f} µm, R^2={r2:.4f}")
print("Normal fit on reconstructed samples:")
print(f"  mu={mu_mle:.3f} µm, sigma={sigma_mle:.3f} µm, N={len(samples)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bin_edges = np.array([
    73.0526,  75.3053,  77.5579,  79.8105,  82.0632,
    84.3158,  86.5684,  88.8211,  91.0737,  93.3263,
    95.5789,  97.8316, 100.0842, 102.3368, 104.5895,
   106.8421, 109.0947, 111.3474, 113.6000, 115.8526,
   118.1053
])

hist_heights = np.array([
    0.0148, 0.0106, 0.0129, 0.0144, 0.0228,
    0.0209, 0.0232, 0.0218, 0.0355, 0.0247,
    0.0275, 0.0278, 0.0458, 0.0270, 0.0245,
    0.0158, 0.0220, 0.0137, 0.0201, 0.0163
])

def gauss(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def bimodal(x, A1, mu1, sigma1, A2, mu2, sigma2):
    return (
        A1 * np.exp(-0.5 * ((x - mu1) / sigma1) ** 2) +
        A2 * np.exp(-0.5 * ((x - mu2) / sigma2) ** 2)
    )

# one-Gaussian params
A1 = 0.0345
mu1 = 97.18
sigma1 = 12.35

# bimodal params
A2_1 = 0.00868
mu2_1 = 83.94
sigma2_1 = 4.30

A2_2 = 0.03025
mu2_2 = 101.29
sigma2_2 = 14.62

x = np.linspace(bin_edges[0], bin_edges[-1], 500)

plt.figure(figsize=(6.4, 4.0))
plt.bar(
    bin_edges[:-1],
    hist_heights,
    width=np.diff(bin_edges),
    align="edge",
    alpha=0.5
)
plt.plot(x, gauss(x, A1, mu1, sigma1), label="1 Gaussian")
plt.plot(
    x,
    bimodal(x, A2_1, mu2_1, sigma2_1, A2_2, mu2_2, sigma2_2),
    label="Bimodal"
)

plt.xlim(72, 120)
plt.ylim(0, 0.047)
plt.title("Intensity diagram of droplets (iLOV), 30um")
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

# counts: counts per bin
# bin_edges: length = len(counts)+1

counts = np.asarray(counts, dtype=float)
bin_edges = np.asarray(bin_edges, dtype=float)
n = counts.sum()

def bin_probs_gauss(params, edges):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    cdf_vals = norm.cdf(edges, loc=mu, scale=sigma)
    p = np.diff(cdf_vals)
    return np.clip(p, 1e-12, 1.0)

def neg_logL_gauss(params, counts, edges):
    p = bin_probs_gauss(params, edges)
    return -np.sum(counts * np.log(p))

# initial guess
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
mu0 = np.average(bin_centers, weights=counts)
sigma0 = np.sqrt(np.average((bin_centers - mu0)**2, weights=counts))

res1 = minimize(
    neg_logL_gauss,
    x0=[mu0, np.log(sigma0)],
    args=(counts, bin_edges),
    method='Nelder-Mead'
)

logL1 = -res1.fun
mu1, log_sigma1 = res1.x
sigma1 = np.exp(log_sigma1)
k1 = 2
AIC1 = 2 * k1 - 2 * logL1
BIC1 = k1 * np.log(n) - 2 * logL1

print("1 Gaussian")
print("mu =", mu1, "sigma =", sigma1)
print("logL =", logL1, "AIC =", AIC1, "BIC =", BIC1)